#  The ”Budget Keeper” (Token Economics)

#### Users are forwarding long chain-spam messages

In [11]:
import sys
sys.path.append('..')


from utils.token_utils import count_messages_tokens
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
TOKEN_LIMIT = 150

model = pick_model('google', 'general')
client = LLMClient('google', model)

In [19]:
spam_message = (
    "FWD: FWD: FWD: FWD: URGENT HELP NEEDED!!! "
    "Please share this message with everyone you know! "
    "There has been a massive flood in the southern province. "
    "Thousands of families are displaced and need immediate assistance. "
    "The government has declared a state of emergency. "
    "The army, navy, and air force have been deployed to rescue stranded people. "
    "If you know anyone in the affected areas, please contact them immediately. "
    "Donation centers have been set up at the following locations: "
    "Galle Town Hall, Matara Public Library, Hambantota Community Center, "
    "and Tangalle Temple. They need rice, water, clothes, and medicine. "
    "The Red Cross and other NGOs are also on the ground providing aid. "
    "Please do not spread rumors or misinformation. "
    "Only share verified information from official sources. "
    "Stay safe and help each other during this difficult time. "
    "Forward this to at least 10 people to spread awareness! "
    "God bless Sri Lanka. 🙏🙏🙏 #FloodRelief #HelpSriLanka "
    "FWD: FWD: FWD: This message has been forwarded many times. "
    "Original sender unknown. Please verify before acting on this information."
)

examples = """

Example 1:
Message: We are trapped on the roof with 3 children and water is rising fast.
District: None
Intent: Rescue
Priority: High
Explanation: This is a life-threatening emergency with people trapped and immediate danger.

Example 2:
Message: Need food packets and drinking water at school shelter in Gampaha.
District: Gampaha
Intent: Supply
Priority: High
Explanation: The message requests essential supplies but does not indicate immediate danger to life.

Example 3:
Message: Breaking News: Kelani River level at 9m, flood warning issued.
District: Colombo
Intent: Info
Priority: Low
Explanation: This is a news report providing informational updates, not a direct request for help.

Example 4:
Message: Electricity restored in Kalutara town after floods.
District: Kalutara
Intent: Other
Priority: Low
Explanation: This is a general status update and does not require emergency response.

"""

normal_message = "SOS: 5 people trapped on a roof in Ja-Ela. Need boat immediately."

def budget_keeper(message:str) -> dict:
    """
    Token guard: count tokens, then truncate/summarize if over budget.

    Returns a dict with:
        status : "ALLOWED" | "BLOCKED/TRUNCATED"
        tokens : original token count
        action : what was done
        response : LLM classification result (or none if blocked)
    """

    messages = [{'role':'user','content':message}]
    token_info = count_messages_tokens(messages,'google',model)
    token_count = token_info['estimated_total']

    print(f"Token count: {token_count} | {TOKEN_LIMIT} limit")

    if token_count <= TOKEN_LIMIT:
        print("ALLOWED - within token budget.\n")

        prompt_text, spec = render(
            "few_shot.v1",
            role = "crisis message classifier",
            examples = examples,
            query = f"Message: {message}\n",
            constraints = (
                "Classify the message strictly using the learned patterns."
                "Do not hallucinate any information."
                "Search the district and add it if it in the messege."
                "Because most of messages with district."
                "If district is not mentioned, respond with 'None'."
            ),
            format=(
                "District: [Name] | "
                "Intent: [Rescue/Supply/Info/Other] | "
                "Priority: [High/Low]"
            )
        )
        
        msgs = [{"role": "user", "content": prompt_text}]

        response = client.chat(
            msgs,
            temperature=0.0,
        )

        return {
            "status": "ALLOWED",
            "tokens": token_count,
            "action": "Processed normally",
            "response": response['text'].strip(),
        }

    else:
        print("BLOCKED/TRUNCATED - exceeded token budget!")
        print(f"Overflowed by {token_count - TOKEN_LIMIT} tokens.")
        print("Applying overflow summaize to condense before processing....\n")


        prompt_text, spec = render(
            "overflow_summarize.v1",
            max_tokens_context="80",
            context=message,
            task=(
                "Classify this crisis message.\n"
                "Output: District: [Name] | Intent: [Rescue/Supply/Info/Other] | Priority: [High/Low]"
            ),
            format="District: [Name] | Intent: [Rescue/Supply/Info/Other] | Priority: [High/Low]",
        )

        response = client.chat(
            [{"role": "user", "content": prompt_text}],
            temperature=spec.temperature or 0.2,
        )

        return {
            "status": "BLOCKED/TRUNCATED",
            "tokens": token_count,
            "action": "Summarized via overflow_summarize.v1 before processing",
            "response": response['text'].strip(),
        }

In [20]:
if __name__ == "__main__":
    print("="*60)
    print("Spam Detection")
    print("="*60)

    print("\n ---- SPAM / LONG CHAIN MESSAGE ----\n")
    print(f"Message preview:{spam_message[:80]}...")
    result = budget_keeper(spam_message)
    print(f"Status: {result['status']}")
    print(f"Tokens: {result['tokens']}")
    print(f"Action: {result['action']}")
    print(f"Response: {result['response']}")


    print("\n ---- NORMAL SHORT MESSAGE ----\n")
    print(f"Message preview:{normal_message}...")
    result = budget_keeper(normal_message)
    print(f"Status: {result['status']}")
    print(f"Tokens: {result['tokens']}")
    print(f"Action: {result['action']}")
    print(f"Response: {result['response']}")

Spam Detection

 ---- SPAM / LONG CHAIN MESSAGE ----

Message preview:FWD: FWD: FWD: FWD: URGENT HELP NEEDED!!! Please share this message with everyon...
Token count: 238 | 150 limit
BLOCKED/TRUNCATED - exceeded token budget!
Overflowed by 88 tokens.
Applying overflow summaize to condense before processing....

Status: BLOCKED/TRUNCATED
Tokens: 238
Action: Summarized via overflow_summarize.v1 before processing
Response: Step 1 Summary:
Massive flood in southern province displaces thousands. State of emergency declared. Army, navy, air force deployed for rescue. Donation centers: Galle Town Hall, Matara Public Library, Hambantota Community Center, Tangalle Temple. Needs: rice, water, clothes, medicine. Red Cross & NGOs providing aid. Verify info.

Step 2 Classification:
District: Southern Province | Intent: Rescue/Supply | Priority: High

 ---- NORMAL SHORT MESSAGE ----

Message preview:SOS: 5 people trapped on a roof in Ja-Ela. Need boat immediately....
Token count: 25 | 150 limit
ALLO